### Install the proper libraries

In [ ]:
!pip install -q transformers datasets evaluate accelerate scikit-learn
!pip install -U transformers huggingface_hub

### Import Libraries, Device & Environment Checks

In [ ]:
import os
import random
import pandas as pd
import numpy as np
import json
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import inspect
import transformers
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback
import inspect
from transformers import TrainingArguments
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    set_seed
)
import evaluate

# Reproducibility
RANDOM_SEED = 42
set_seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# Device / GPU check
print("="*70)
print("PyTorch version:", torch.__version__)
n_gpus = torch.cuda.device_count()
print("Number of CUDA GPUs available:", n_gpus)
if n_gpus > 0:
    print("Current CUDA device:", torch.cuda.get_device_name(0))
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
print("="*70)

### EDA

In [ ]:
# Load Dataset and make the unique id on the data the kaggle's id
csv_path = "/kaggle/input/sentiment-analysis-for-mental-health/Combined Data.csv"
df = pd.read_csv(csv_path, index_col=0)

# the shape and the list of the columns
print("Rows, columns:", df.shape)
print("Columns:", df.columns.tolist())
print("="*70)
display(df.head())
print("="*70)

In [ ]:
# Display the missing values in data
print("="*70)
print("\nMissing values per column:")
display(df.isnull().sum().reset_index())
print("="*70)

In [ ]:
# drop missing data in our data and verify
print("="*70)
df = df.dropna()
display(df.isnull().sum().reset_index())
display(df.shape)
print("="*70)

In [ ]:
# lets see the unique values in status and count them
print("="*70)
display(df['status'].value_counts().reset_index())
print("="*70)

In [ ]:
# Gives a quick overview of how the dataset is distributed across sentiment classes.
print("="*70)
sentiment_counts = df["status"].value_counts()

plt.figure(figsize=(8, 8))
plt.pie(sentiment_counts, labels=sentiment_counts.index, autopct='%1.1f%%', startangle=140)
plt.title("Sentiment Label Proportions")
plt.axis("equal")
plt.show()
print("="*70)

In [ ]:
# word cloud shows the most frequent words used in statements for each sentiment class.
# the bigger & bolder the text the more frequently it wa smentioned the the statement status.
print("="*70)
for label in df["status"].unique():
    text = " ".join(df[df["status"] == label]["statement"])
    wordcloud = WordCloud(width=800, height=400, background_color='white').generate(text)
    
    plt.figure(figsize=(8, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis("off")
    plt.title(f"Word Cloud for {label}")
    plt.show()
    print("="*70)

In [ ]:
# shows how long the statement text length for each status or sentiment and create the column
# suicidal has much text length
print("="*70)
df["text_length"] = df["statement"].apply(lambda x: len(str(x).split()))

plt.figure(figsize=(10, 6))
sns.violinplot(data=df, x="status", y="text_length", inner="quartile")
plt.title("Violin Plot of Text Length by Sentiment")
plt.xticks(rotation=45)
plt.show()
print("="*70)

In [ ]:
# Shows which sentiment categories tend to have longer or shorter statements on average.
print("="*70)
avg_lengths = df.groupby("status")["text_length"].mean().sort_values()

plt.figure(figsize=(10, 6))
sns.barplot(x=avg_lengths.index, y=avg_lengths.values)
plt.title("Average Text Length per Sentiment")
plt.ylabel("Average Word Count")
plt.xticks(rotation=45)
plt.show()
print("="*70)

In [ ]:
# Trim whitespace in text and labels if theres any
print("="*70)
df["statement"] = df["statement"].astype(str).str.strip()
df["status"] = df["status"].astype(str).str.strip()
df.head()

# Lets drop the text_length column
if "text_length" in df.columns:
    df = df.drop(columns=["text_length"])
    print("Dropped 'text_length' column. Current columns:", df.columns.tolist())
else:
    print("'text_length' not found — nothing to drop.")

df.head()
print("="*70)

### Label Encoder

In [ ]:
# we encode the labels to integer class ids using scikit learn's label encoder
print("="*70)
le = LabelEncoder()
df['label'] = le.fit_transform(df["status"])
df['label'].unique()

# Save mapping for later decoding predictions (int -> string)
label_mapping = {int(i): label for i, label in enumerate(le.classes_)}
print("\nLabel encoder mapping (int -> label):")
print(label_mapping)

# Verify the label column
display(df.head())

# Save number of classes variable for model init - so we can convert a label number
# back to status name like depression etc
num_labels = len(le.classes_)
print("\nNumber of hi classes (num_labels) =", num_labels)
print("="*70)

In [ ]:
# Save mapping to disk - helpful when downloading our model artifacts (this is optional btw)
print("="*70)
import json
mapping_path = "./label_mapping.json"
with open(mapping_path, "w") as f:
    json.dump(label_mapping, f, indent=2)
print("Saved label mapping to", mapping_path)
print("="*70)

### Train/Validation/Test Split (Stratified) - 80/10/10

In [ ]:
# we use stratified splitting to preserve class proportions and it avoids bias,
# giving same proportions to each split
print("="*70)
train_df, temp_df = train_test_split(df, test_size=0.20, 
                                     stratify=df["label"], random_state=RANDOM_SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, 
                                   stratify=temp_df["label"], random_state=RANDOM_SEED)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)
print("="*70)

### Convert pandas splits → Hugging Face DatasetDict

In [ ]:
# Convert to HF Dataset and DatasetDict
# Trainer works nicely with HF Dataset objects.
print("="*70)
train_ds = Dataset.from_pandas(train_df[["statement", "label"]].reset_index(drop=True))
val_ds = Dataset.from_pandas(val_df[["statement", "label"]].reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df[["statement", "label"]].reset_index(drop=True))

hf_datasets = DatasetDict({"train": train_ds, "validation": val_ds, "test": test_ds})
print(hf_datasets)
print("="*70)

### Initialize tokenizer and model (DistilBERT)

In [ ]:
# This is where DistilBERT is instantiated for tokenization and classification.
# tokenizer & model init
print("="*70)
MODEL_NAME = "distilbert-base-uncased"
tokenizer = DistilBertTokenizerFast.from_pretrained(
    MODEL_NAME,
    trust_remote_code=False
)
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=num_labels,
    trust_remote_code=False
)
model.to("cuda" if torch.cuda.is_available() else "cpu")

print("="*70)

### Tokenize the datasets

In [ ]:
# We tokenise all splits once (batched) and remove the original text column to save memory.
print("="*70)
MAX_LENGTH = 128 # safe default for our sentence-level data, reduce if you need memory.

def tokenize_fn(batch):
    return tokenizer(batch["statement"], truncation=True, padding=False, max_length=MAX_LENGTH)

# Map tokenization (batched)
tokenized = hf_datasets.map(tokenize_fn, batched=True, batch_size=512)

# Remove original 'statement' column (our trainer does not need raw text) and set format
tokenized = tokenized.remove_columns(["statement"])
tokenized.set_format(type="torch")
print(tokenized)
print("="*70)

### Data collator & metrics

In [ ]:
# data collator and metrics - pads each batch to the longest sample in that batch.
# this is memory efficient
print("="*70)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy.compute(predictions=preds, references=labels)
    f1_weighted = f1.compute(predictions=preds, references=labels, average="weighted")
    return {"accuracy": acc["accuracy"], "f1_weighted": f1_weighted["f1"]}

print("Metrics configured: accuracy and weighted F1-score")
print("="*70)

### TrainingArguments & Trainer configuration

In [ ]:
# Our desired hyperparameters
print("="*70)
OUTPUT_DIR = "./distilbert_mh_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# OPTIMIZED HYPERPARAMETERS
PER_DEVICE_BATCH_SIZE = 32
NUM_EPOCHS = 4
LEARNING_RATE = 4e-5
WEIGHT_DECAY = 0.01

print(f"\nTraining Configuration:")
print(f"  Batch size: {PER_DEVICE_BATCH_SIZE}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Weight decay: {WEIGHT_DECAY}")
print("="*70)

In [ ]:
# First, let's check which parameters our version supports
print("="*70)
sig = inspect.signature(TrainingArguments.__init__)
supported_params = set(sig.parameters.keys())
supported_params.discard("self")

print("Checking supported parameters in our transformers version...")
print(f"Supports 'evaluation_strategy': {'evaluation_strategy' in supported_params}")
print(f"Supports 'eval_strategy': {'eval_strategy' in supported_params}")
print(f"Supports 'save_strategy': {'save_strategy' in supported_params}")

# Build training arguments dictionary
training_args_dict = {
    "output_dir": OUTPUT_DIR,
    "num_train_epochs": NUM_EPOCHS,
    "per_device_train_batch_size": PER_DEVICE_BATCH_SIZE,
    "per_device_eval_batch_size": PER_DEVICE_BATCH_SIZE * 2,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "logging_dir": "./logs",
    "logging_steps": 50,
    "logging_first_step": True,
    "load_best_model_at_end": True,
    "metric_for_best_model": "f1_weighted",
    "greater_is_better": True,
    "save_total_limit": 2,
    "seed": RANDOM_SEED,
    "disable_tqdm": False,
    "report_to": "none",
}

print()
# we add optional optimizations (if supported)
if "gradient_accumulation_steps" in supported_params:
    training_args_dict["gradient_accumulation_steps"] = 2
    print("Added gradient_accumulation_steps")

if "fp16" in supported_params and torch.cuda.is_available():
    training_args_dict["fp16"] = True
    print("Added fp16 (mixed precision training)")

if "dataloader_num_workers" in supported_params:
    training_args_dict["dataloader_num_workers"] = 2
    print("Added dataloader_num_workers")

if "warmup_ratio" in supported_params:
    training_args_dict["warmup_ratio"] = 0.1
    print("Added warmup_ratio")

# Handle evaluation strategy (version-dependent)
if "eval_strategy" in supported_params:
    training_args_dict["eval_strategy"] = "steps"
    training_args_dict["eval_steps"] = 200
    print("Using 'eval_strategy' (newer version)")
elif "evaluation_strategy" in supported_params:
    training_args_dict["evaluation_strategy"] = "steps"
    training_args_dict["eval_steps"] = 200
    print("Using 'evaluation_strategy' (older version)")
else:
    print("Strategy params not found, will evaluate per epoch by default")
if "save_strategy" in supported_params:
    training_args_dict["save_strategy"] = "steps"
    training_args_dict["save_steps"] = 200
    print("Using 'save_strategy'")
elif "save_steps" in supported_params:
    training_args_dict["save_steps"] = 200
    print("Using 'save_steps'")

# Create TrainingArguments with compatible parameters
training_args = TrainingArguments(**training_args_dict)

print("\n" + "="*70)
print("TrainingArguments created successfully!")
print("="*70)
print(f"Output directory: {OUTPUT_DIR}")
print(f"Batch size: {PER_DEVICE_BATCH_SIZE}")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Expected steps per epoch: ~{len(tokenized['train']) // (PER_DEVICE_BATCH_SIZE * training_args_dict.get('gradient_accumulation_steps', 1))}")
print(f"Total training steps: ~{(len(tokenized['train']) // (PER_DEVICE_BATCH_SIZE * training_args_dict.get('gradient_accumulation_steps', 1))) * NUM_EPOCHS}")
print("="*70 + "\n")

### Create the Trainer with Early Stopping

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=3,  # Stop if no improvement for 3 evaluations
            early_stopping_threshold=0.001  # Minimum improvement (0.1%)
        )
    ]
)

print("="*70)
print("TRAINER INITIALIZED")
print("="*70)
print(f"✓ Training samples: {len(tokenized['train']):,}")
print(f"✓ Validation samples: {len(tokenized['validation']):,}")
print(f"✓ Test samples: {len(tokenized['test']):,}")
print(f"✓ Model device: {model.device}")
print(f"✓ Model parameters: {model.num_parameters():,}")
print(f"✓ Early stopping: Enabled (patience=3)")
print("="*70 + "\n")

### Pre-Training Checks and Information

In [ ]:
print("="*70)
print("PRE-TRAINING INFORMATION")
print("="*70)

# Calculate training statistics
steps_per_epoch = len(tokenized['train']) // (PER_DEVICE_BATCH_SIZE * training_args_dict.get('gradient_accumulation_steps', 1))
total_steps = steps_per_epoch * NUM_EPOCHS
eval_frequency = training_args_dict.get('eval_steps', steps_per_epoch)
num_evaluations = total_steps // eval_frequency

print(f"\n Training Statistics:")
print(f"  • Steps per epoch: {steps_per_epoch}")
print(f"  • Total training steps: {total_steps}")
print(f"  • Evaluation frequency: Every {eval_frequency} steps")
print(f"  • Expected evaluations: ~{num_evaluations}")
print(f"  • Warmup steps: {int(total_steps * training_args_dict.get('warmup_ratio', 0))}")

print(f"\n  Training Configuration:")
print(f"  • Effective batch size: {PER_DEVICE_BATCH_SIZE} × {training_args_dict.get('gradient_accumulation_steps', 1)} = {PER_DEVICE_BATCH_SIZE * training_args_dict.get('gradient_accumulation_steps', 1)}")
print(f"  • Learning rate: {LEARNING_RATE}")
print(f"  • Weight decay: {WEIGHT_DECAY}")
print(f"  • Mixed precision (FP16): {training_args_dict.get('fp16', False)}")

print(f"\n Classes to predict:")
for idx, label in label_mapping.items():
    count = (df['label'] == idx).sum()
    percentage = (count / len(df)) * 100
    print(f"  • {idx}: {label:20s} ({count:,} samples, {percentage:.1f}%)")

print(f"\n Estimated training time:")
print(f"  • With GPU (T4/P100): ~15-25 minutes")
print(f"  • With GPU (V100/A100): ~8-15 minutes")
print(f"  • Without GPU: ~2-4 hours (not recommended)")
print("="*70)

### Start Training with Progress Monitoring

In [ ]:
import time

print("="*70)
print("STARTING TRAINING")
print("="*70)
print(f"Start time: {time.strftime('%Y-%m-%d %H:%M:%S')}")
print("="*70 + "\n")

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU cache cleared")

# Start training
start_time = time.time()

try:
    train_result = trainer.train()
    training_successful = True
except KeyboardInterrupt:
    print("\n\n Training interrupted by user!")
    training_successful = False
except Exception as e:
    print(f"\n\n Training failed with error: {e}")
    training_successful = False
    raise

end_time = time.time()
training_duration = end_time - start_time

# Display results
print("\n" + "="*70)
if training_successful:
    print("TRAINING COMPLETED SUCCESSFULLY!")
else:
    print("TRAINING INCOMPLETE")
print("="*70)

if training_successful:
    print(f"\n Training Summary:")
    print(f"  • Final training loss: {train_result.training_loss:.4f}")
    print(f"  • Total training time: {training_duration:.2f} seconds ({training_duration/60:.2f} minutes)")
    print(f"  • Training speed: {train_result.metrics['train_samples_per_second']:.2f} samples/sec")
    print(f"  • Steps completed: {train_result.global_step}")
    
    # Check if early stopping was triggered
    if train_result.global_step < total_steps:
        print(f"  • Early stopping triggered at step {train_result.global_step}/{total_steps}")
        print(f"  • Time saved: ~{((total_steps - train_result.global_step) / steps_per_epoch):.1f} epochs")
    else:
        print(f"  • Completed all {NUM_EPOCHS} epochs")

print("="*70 + "\n")

# Visualise our training
if training_successful:
    import matplotlib.pyplot as plt
    
    logs = trainer.state.log_history
    eval_steps = [log['step'] for log in logs if 'eval_accuracy' in log]
    eval_loss = [log['eval_loss'] for log in logs if 'eval_loss' in log]
    eval_acc = [log['eval_accuracy'] for log in logs if 'eval_accuracy' in log]
    eval_f1 = [log['eval_f1_weighted'] for log in logs if 'eval_f1_weighted' in log]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    ax1.plot(eval_steps, eval_loss, 'r-o', linewidth=2, markersize=6)
    ax1.set_title('Validation Loss', fontsize=13, fontweight='bold')
    ax1.set_xlabel('Steps')
    ax1.set_ylabel('Loss')
    ax1.grid(True, alpha=0.3)
 
    ax2.plot(eval_steps, eval_acc, 'g-s', linewidth=2, markersize=6, label='Accuracy')
    ax2.plot(eval_steps, eval_f1, 'm-^', linewidth=2, markersize=6, label='F1-Score')
    ax2.set_title('Validation Metrics', fontsize=13, fontweight='bold')
    ax2.set_xlabel('Steps')
    ax2.set_ylabel('Score')
    ax2.set_ylim([0.7, 1.0])
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/training_plot.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Plot saved to: {OUTPUT_DIR}/training_plot.png\n")

### Evaluate on Validation Set

In [ ]:
if training_successful:
    print("="*70)
    print("EVALUATING ON VALIDATION SET")
    print("="*70)
    
    val_results = trainer.evaluate(tokenized["validation"])
    
    print("\nValidation Results:")
    print(f"  • Loss: {val_results['eval_loss']:.4f}")
    print(f"  • Accuracy: {val_results['eval_accuracy']:.4f} ({val_results['eval_accuracy']*100:.2f}%)")
    print(f"  • F1-Score (weighted): {val_results['eval_f1_weighted']:.4f}")
    print(f"  • Evaluation time: {val_results['eval_runtime']:.2f} seconds")
    print(f"  • Evaluation speed: {val_results['eval_samples_per_second']:.2f} samples/sec")
    
    print("\n" + "="*70 + "\n")
else:
    print("Skipping validation evaluation (training incomplete)\n")

### Evaluate on Test Set (Final Performance)

In [ ]:
if training_successful:
    print("="*70)
    print("EVALUATING ON TEST SET (FINAL PERFORMANCE)")
    print("="*70)
    print("This is the unbiased estimate of real-world performance.\n")
    
    test_results = trainer.evaluate(tokenized["test"])
    
    print("Test Results:")
    print(f"  • Loss: {test_results['eval_loss']:.4f}")
    print(f"  • Accuracy: {test_results['eval_accuracy']:.4f} ({test_results['eval_accuracy']*100:.2f}%)")
    print(f"  • F1-Score (weighted): {test_results['eval_f1_weighted']:.4f}")
    
    # Compare with validation
    val_acc = val_results['eval_accuracy']
    test_acc = test_results['eval_accuracy']
    difference = abs(val_acc - test_acc)
    
    print(f"\nValidation vs Test:")
    print(f"  • Validation accuracy: {val_acc:.4f}")
    print(f"  • Test accuracy: {test_acc:.4f}")
    print(f"  • Difference: {difference:.4f} ({difference*100:.2f}%)")
    
    if difference < 0.02:
        print("  • Model generalizes well (difference < 2%)")
    elif difference < 0.05:
        print("  • Slight overfitting (difference 2-5%)")
    else:
        print("  • Significant overfitting (difference > 5%)")
    
    print("\n" + "="*70 + "\n")

# Data for visualization
    metrics = ['Accuracy', 'F1-Score', 'Loss']
    val_values = [val_results['eval_accuracy'], val_results['eval_f1_weighted'], val_results['eval_loss']]
    test_values = [test_results['eval_accuracy'], test_results['eval_f1_weighted'], test_results['eval_loss']]
    

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    

    x = np.arange(2)
    width = 0.35
    
    ax1 = axes[0]
    bars1 = ax1.bar(x - width/2, [val_acc, val_results['eval_f1_weighted']], 
                    width, label='Validation', color='skyblue', edgecolor='black')
    bars2 = ax1.bar(x + width/2, [test_acc, test_results['eval_f1_weighted']], 
                    width, label='Test', color='lightcoral', edgecolor='black')
    
    ax1.set_ylabel('Score', fontsize=12)
    ax1.set_title('Validation vs Test Performance', fontsize=14, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(['Accuracy', 'F1-Score'])
    ax1.set_ylim([0.75, 1.0])
    ax1.legend()
    ax1.grid(True, axis='y', alpha=0.3)
    
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax1.text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.3f}', ha='center', va='bottom', fontsize=10)
    
    ax2 = axes[1]
    bars = ax2.bar(['Validation', 'Test'], 
                   [val_results['eval_loss'], test_results['eval_loss']],
                   color=['skyblue', 'lightcoral'], edgecolor='black')
    
    ax2.set_ylabel('Loss', fontsize=12)
    ax2.set_title('Loss Comparison', fontsize=14, fontweight='bold')
    ax2.grid(True, axis='y', alpha=0.3)
    
    for bar in bars:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.4f}', ha='center', va='bottom', fontsize=11)
    
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/test_results.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Test results plot saved: {OUTPUT_DIR}/test_results.png\n")
    
else:
    print("Skipping test evaluation (training incomplete)\n")

### Save Model and All Results

In [ ]:
if training_successful:
    print("="*70)
    print("SAVING MODEL AND RESULTS")
    print("="*70)
    
    # Save model and tokenizer
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    
    print(f"Model saved to: {OUTPUT_DIR}/pytorch_model.bin")
    print(f"Config saved to: {OUTPUT_DIR}/config.json")
    print(f"Tokenizer saved to: {OUTPUT_DIR}/")
    
    # Create comprehensive results dictionary
    results_summary = {
        "model_info": {
            "model_name": MODEL_NAME,
            "num_labels": num_labels,
            "label_mapping": label_mapping,
            "max_length": MAX_LENGTH
        },
        "dataset_info": {
            "total_samples": len(df),
            "train_samples": len(tokenized['train']),
            "validation_samples": len(tokenized['validation']),
            "test_samples": len(tokenized['test'])
        },
        "training_config": {
            "batch_size": PER_DEVICE_BATCH_SIZE,
            "gradient_accumulation_steps": training_args_dict.get('gradient_accumulation_steps', 1),
            "effective_batch_size": PER_DEVICE_BATCH_SIZE * training_args_dict.get('gradient_accumulation_steps', 1),
            "epochs": NUM_EPOCHS,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "warmup_ratio": training_args_dict.get('warmup_ratio', 0),
            "fp16": training_args_dict.get('fp16', False)
        },
        "training_results": {
            "final_loss": float(train_result.training_loss),
            "total_steps": int(train_result.global_step),
            "training_time_seconds": float(training_duration),
            "training_time_minutes": float(training_duration / 60),
            "samples_per_second": float(train_result.metrics['train_samples_per_second']),
            "early_stopped": train_result.global_step < total_steps
        },
        "validation_results": {
            "loss": float(val_results['eval_loss']),
            "accuracy": float(val_results['eval_accuracy']),
            "f1_weighted": float(val_results['eval_f1_weighted'])
        },
        "test_results": {
            "loss": float(test_results['eval_loss']),
            "accuracy": float(test_results['eval_accuracy']),
            "f1_weighted": float(test_results['eval_f1_weighted'])
        },
        "timestamp": time.strftime('%Y-%m-%d %H:%M:%S')
    }
    
    # Save results as JSON
    results_path = f"{OUTPUT_DIR}/training_results.json"
    with open(results_path, "w") as f:
        json.dump(results_summary, f, indent=2)
    
    print(f"Results saved to: {results_path}")
    
    # Save label mapping separately for easy access
    label_mapping_path = f"{OUTPUT_DIR}/label_mapping.json"
    with open(label_mapping_path, "w") as f:
        json.dump(label_mapping, f, indent=2)
    
    print(f"Label mapping saved to: {label_mapping_path}")
    
    print("\n" + "="*70 + "\n")
else:
    print("Skipping model save (training incomplete)\n")

### Test with Real-World Examples

In [ ]:
if training_successful:
    print("="*70)
    print("TESTING WITH REAL-WORLD EXAMPLES")
    print("="*70)
    
    # Test samples covering all classes
    test_samples = [
        "I feel so anxious and worried all the time, can't calm down my racing thoughts",
        "I am feeling great and very happy today! Life is wonderful",
        "I have thoughts of ending my life, nothing matters anymore",
        "My mood swings are extreme, one moment happy then suddenly very sad and angry",
        "I feel so worthless and empty, nothing brings me joy anymore",
        "I can't control my anger and my behavior is unpredictable",
        "Work is stressful but I'm managing it well, just need some rest",
        "Just a normal day, nothing special happening, feeling okay"
    ]
    
    # Set model to evaluation mode
    model.eval()
    
    print("\nPredictions:\n")
    
    for i, text in enumerate(test_samples, 1):
        # Tokenize
        inputs = tokenizer(
            text, 
            return_tensors="pt", 
            truncation=True, 
            max_length=MAX_LENGTH,
            padding=True
        )
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        
        # Predict
        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
            pred_id = torch.argmax(probs, dim=-1).item()
            confidence = probs[0][pred_id].item()
            
            # Get top 3 predictions
            top3_probs, top3_ids = torch.topk(probs[0], k=min(3, num_labels))
        
        pred_label = label_mapping[pred_id]
        
        print(f"[{i}] Text: {text[:70]}{'...' if len(text) > 70 else ''}")
        print(f"    Predicted: {pred_label} (confidence: {confidence:.1%})")
        
        # Show top 3 if confidence is low
        if confidence < 0.70:
            print(f"    Top 3 predictions:")
            for prob, idx in zip(top3_probs, top3_ids):
                print(f"       • {label_mapping[idx.item()]}: {prob.item():.1%}")
        print()
    
    print("="*70 + "\n")
else:
    print("Skipping predictions (training incomplete)\n")

### Generate Confusion Matrix (Visual Analysis)

In [ ]:
if training_successful:
    from sklearn.metrics import confusion_matrix, classification_report
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    print("="*70)
    print("GENERATING CONFUSION MATRIX")
    print("="*70)
    
    # Get predictions for entire test set
    print("Generating predictions for test set...")
    predictions = trainer.predict(tokenized["test"])
    pred_labels = np.argmax(predictions.predictions, axis=-1)
    true_labels = predictions.label_ids
    
    # Confusion matrix
    cm = confusion_matrix(true_labels, pred_labels)
    class_names = [label_mapping[i] for i in range(num_labels)]
    
    # Plot
    plt.figure(figsize=(12, 10))
    sns.heatmap(
        cm, 
        annot=True, 
        fmt='d', 
        cmap='Blues',
        xticklabels=class_names, 
        yticklabels=class_names,
        cbar_kws={'label': 'Number of Samples'}
    )
    plt.title('Confusion Matrix - Test Set\n(Higher numbers on diagonal = Better)', fontsize=14, pad=20)
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    
    # Save
    cm_path = f"{OUTPUT_DIR}/confusion_matrix.png"
    plt.savefig(cm_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"Confusion matrix saved to: {cm_path}")
    
    # Classification report (detailed per-class metrics)
    print("\n" + "="*70)
    print("DETAILED CLASSIFICATION REPORT")
    print("="*70)
    print("\nPer-class Performance:\n")
    
    report = classification_report(
        true_labels, 
        pred_labels, 
        target_names=class_names, 
        digits=4,
        output_dict=False
    )
    print(report)
    
    # Save report
    report_dict = classification_report(
        true_labels, 
        pred_labels, 
        target_names=class_names, 
        digits=4,
        output_dict=True
    )
    report_path = f"{OUTPUT_DIR}/classification_report.json"
    with open(report_path, "w") as f:
        json.dump(report_dict, f, indent=2)
    
    print(f"\n Classification report saved to: {report_path}")
    print("\n" + "="*70 + "\n")
else:
    print("Skipping confusion matrix (training incomplete)\n")

### Final Summary

In [ ]:
if training_successful:
    print("="*70)
    print("TRAINING PIPELINE COMPLETED SUCCESSFULLY!")
    print("="*70)
    
    print(f"\nAll files saved to: {OUTPUT_DIR}/")
    print(f"\nFinal Performance:")
    print(f"  • Test Accuracy: {test_results['eval_accuracy']:.2%}")
    print(f"  • Test F1-Score: {test_results['eval_f1_weighted']:.4f}")
    print(f"  • Training Time: {training_duration/60:.1f} minutes")
    
    print(f"\nModel is ready for:")
    print(f"  • Real-time predictions")
    print(f"  • Deployment to production")
    print(f"  • Further fine-tuning")
    print(f"  • Integration into applications")
    
    print(f"\nTo load this model later:")
    print(f"  from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast")
    print(f"  model = DistilBertForSequenceClassification.from_pretrained('{OUTPUT_DIR}')")
    print(f"  tokenizer = DistilBertTokenizerFast.from_pretrained('{OUTPUT_DIR}')")
    
    print("\n" + "="*70)
    print("ALL DONE!")
    print("="*70 + "\n")
else:
    print("="*70)
    print("TRAINING PIPELINE INCOMPLETE")
    print("="*70)
    print("Please review errors above and restart training.")
    print("="*70 + "\n")

### Model Size Analysis

In [ ]:
import os

print("="*70)
print("MODEL SIZE ANALYSIS")
print("="*70)

# our file sizes
model_file = f"{OUTPUT_DIR}/pytorch_model.bin"
if os.path.exists(model_file):
    size_mb = os.path.getsize(model_file) / (1024 * 1024)
    print(f"\nModel File: {size_mb:.2f} MB")

total_size = sum(
    os.path.getsize(os.path.join(dirpath, filename))
    for dirpath, _, filenames in os.walk(OUTPUT_DIR)
    for filename in filenames
)
print(f"Total Directory: {total_size / (1024 * 1024):.2f} MB")

# set our parameters
total_params = model.num_parameters()
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nParameters:")
print(f"  • Total:     {total_params:,}")
print(f"  • Trainable: {trainable_params:,}")

# our memory estimates
size_fp32 = (total_params * 4) / (1024 * 1024)
size_fp16 = (total_params * 2) / (1024 * 1024)

print(f"\nMemory Footprint:")
print(f"  • FP32 (32-bit): {size_fp32:.2f} MB")
print(f"  • FP16 (16-bit): {size_fp16:.2f} MB")

# we comparison with BERT
bert_params = 110_000_000
ratio = bert_params / total_params

print(f"\n vs BERT-base:")
print(f"  • BERT-base:  {bert_params:,} params ({(bert_params * 4)/(1024*1024):.2f} MB)")
print(f"  • Your model: {total_params:,} params ({size_fp32:.2f} MB)")
print(f"  • Reduction:  {ratio:.1f}x smaller ({(ratio-1)*100:.0f}% smaller)")

# File breakdown
print(f"\n File Breakdown:")
for dirpath, _, filenames in os.walk(OUTPUT_DIR):
    for filename in filenames:
        filepath = os.path.join(dirpath, filename)
        file_size = os.path.getsize(filepath) / (1024 * 1024)
        print(f"  • {filename:40s} {file_size:>8.2f} MB")

print("\n" + "="*70)